# 7.2 DPO（Direct Preference Optimization）

DPO绕过奖励模型，直接用偏好数据优化策略模型，简化了对齐流程。它是RLHF最重要的替代方案。

## 为什么需要DPO？

RLHF的问题：
- 需要4个模型（策略、参考、奖励、价值），显存开销大
- PPO训练不稳定，超参敏感
- 奖励模型训练和PPO优化是两个独立阶段，误差累积

DPO的优势：
- 只需2个模型（策略、参考），显存减半
- 训练更稳定，类似SFT的简单训练循环
- 数学上等价于RLHF的最优解

## 核心公式

**DPO损失**：
$$L_{DPO} = -\mathbb{E}[\log \sigma(\beta(\log \frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \log \frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}))]$$

**隐式奖励**：
$$r(x,y) = \beta \log \frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)}$$

- $y_w$: 优选回复，$y_l$: 劣选回复
- $\beta$: 温度参数，控制偏离参考策略的程度
- DPO的隐式奖励等价于RLHF中奖励模型的最优解

## 1. DPO基础实现

DPO的核心实现非常简洁，只需要计算策略模型和参考模型的log概率比值。

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class SimplePolicy(nn.Module):
    def __init__(self, d_input=64, vocab_size=50):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_input, 128), nn.ReLU(),
            nn.LayerNorm(128),
            nn.Linear(128, vocab_size)
        )
    def forward(self, x):
        return self.net(x)

def dpo_loss(policy_logits, ref_logits, y_chosen, y_rejected, beta=0.1):
    log_pi_chosen = F.log_softmax(policy_logits, dim=-1).gather(1, y_chosen.unsqueeze(1)).squeeze(1)
    log_pi_rejected = F.log_softmax(policy_logits, dim=-1).gather(1, y_rejected.unsqueeze(1)).squeeze(1)
    log_ref_chosen = F.log_softmax(ref_logits, dim=-1).gather(1, y_chosen.unsqueeze(1)).squeeze(1)
    log_ref_rejected = F.log_softmax(ref_logits, dim=-1).gather(1, y_rejected.unsqueeze(1)).squeeze(1)

    log_ratio_chosen = log_pi_chosen - log_ref_chosen
    log_ratio_rejected = log_pi_rejected - log_ref_rejected

    loss = -F.logsigmoid(beta * (log_ratio_chosen - log_ratio_rejected)).mean()

    with torch.no_grad():
        chosen_rewards = beta * log_ratio_chosen
        rejected_rewards = beta * log_ratio_rejected
        accuracy = (chosen_rewards > rejected_rewards).float().mean()
        margin = (chosen_rewards - rejected_rewards).mean()

    return loss, {'accuracy': accuracy.item(), 'margin': margin.item(),
                  'chosen_reward': chosen_rewards.mean().item(),
                  'rejected_reward': rejected_rewards.mean().item()}

policy = SimplePolicy()
ref_policy = SimplePolicy()
ref_policy.load_state_dict(policy.state_dict())
for p in ref_policy.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(policy.parameters(), lr=1e-3, weight_decay=0.01)

n_pairs = 64
x = torch.randn(n_pairs, 64)
y_chosen = torch.randint(0, 50, (n_pairs,))
y_rejected = torch.randint(0, 50, (n_pairs,))

print('=== DPO Training ===')
for epoch in range(50):
    policy_logits = policy(x)
    with torch.no_grad():
        ref_logits = ref_policy(x)
    loss, metrics = dpo_loss(policy_logits, ref_logits, y_chosen, y_rejected, beta=0.1)
    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}: Loss={loss.item():.4f}, '
              f'Acc={metrics["accuracy"]:.2%}, Margin={metrics["margin"]:.4f}')

print(f'\nKey: DPO directly optimizes preference pairs without reward model.')

## 2. DPO变体：IPO与cDPO

标准DPO在某些情况下可能过于激进，导致策略偏离参考模型太远。IPO和cDPO是两种重要的改进变体。

### IPO（Identity Preference Optimization）
- 用平方损失替代log-sigmoid，避免DPO对高置信度偏好的过度优化
- $L_{IPO} = \mathbb{E}[(\beta^{-1}(\log \frac{\pi_\theta(y_w)}{\pi_{ref}(y_w)} - \log \frac{\pi_\theta(y_l)}{\pi_{ref}(y_l)}) - 1)^2]$
- 更保守，适合偏好数据有噪声的情况

### cDPO（Conditional DPO）
- 考虑偏好数据中的标注噪声，引入噪声参数$\epsilon$
- 以$1-\epsilon$的概率y_w确实优于y_l，以$\epsilon$的概率相反
- $L_{cDPO} = -\mathbb{E}[\log \sigma(\beta(\log \text{ratio}_w - \log \text{ratio}_l)) \cdot (1-\epsilon) + \log \sigma(-\beta(\log \text{ratio}_w - \log \text{ratio}_l)) \cdot \epsilon]$

In [ ]:
def ipo_loss(policy_logits, ref_logits, y_chosen, y_rejected, beta=0.1):
    log_pi_chosen = F.log_softmax(policy_logits, dim=-1).gather(1, y_chosen.unsqueeze(1)).squeeze(1)
    log_pi_rejected = F.log_softmax(policy_logits, dim=-1).gather(1, y_rejected.unsqueeze(1)).squeeze(1)
    log_ref_chosen = F.log_softmax(ref_logits, dim=-1).gather(1, y_chosen.unsqueeze(1)).squeeze(1)
    log_ref_rejected = F.log_softmax(ref_logits, dim=-1).gather(1, y_rejected.unsqueeze(1)).squeeze(1)

    log_ratio_chosen = log_pi_chosen - log_ref_chosen
    log_ratio_rejected = log_pi_rejected - log_ref_rejected

    loss = ((log_ratio_chosen - log_ratio_rejected - 1.0 / beta) ** 2).mean()
    return loss

def cdpo_loss(policy_logits, ref_logits, y_chosen, y_rejected, beta=0.1, epsilon=0.1):
    log_pi_chosen = F.log_softmax(policy_logits, dim=-1).gather(1, y_chosen.unsqueeze(1)).squeeze(1)
    log_pi_rejected = F.log_softmax(policy_logits, dim=-1).gather(1, y_rejected.unsqueeze(1)).squeeze(1)
    log_ref_chosen = F.log_softmax(ref_logits, dim=-1).gather(1, y_chosen.unsqueeze(1)).squeeze(1)
    log_ref_rejected = F.log_softmax(ref_logits, dim=-1).gather(1, y_rejected.unsqueeze(1)).squeeze(1)

    log_ratio_chosen = log_pi_chosen - log_ref_chosen
    log_ratio_rejected = log_pi_rejected - log_ref_rejected

    diff = beta * (log_ratio_chosen - log_ratio_rejected)
    loss_correct = -F.logsigmoid(diff)
    loss_flipped = -F.logsigmoid(-diff)
    loss = ((1 - epsilon) * loss_correct + epsilon * loss_flipped).mean()
    return loss

policy_dpo = SimplePolicy()
policy_ipo = SimplePolicy()
policy_cdpo = SimplePolicy()
ref = SimplePolicy()
ref.load_state_dict(policy_dpo.state_dict())
policy_ipo.load_state_dict(policy_dpo.state_dict())
policy_cdpo.load_state_dict(policy_dpo.state_dict())
for p in ref.parameters():
    p.requires_grad = False

opt_dpo = torch.optim.AdamW(policy_dpo.parameters(), lr=1e-3)
opt_ipo = torch.optim.AdamW(policy_ipo.parameters(), lr=1e-3)
opt_cdpo = torch.optim.AdamW(policy_cdpo.parameters(), lr=1e-3)

x = torch.randn(32, 64)
y_w = torch.randint(0, 50, (32,))
y_l = torch.randint(0, 50, (32,))

print('=== DPO Variants Comparison ===')
for epoch in range(50):
    with torch.no_grad():
        ref_logits = ref(x)

    logits_dpo = policy_dpo(x)
    loss_dpo, _ = dpo_loss(logits_dpo, ref_logits, y_w, y_l, beta=0.1)
    opt_dpo.zero_grad()
    loss_dpo.backward()
    opt_dpo.step()

    logits_ipo = policy_ipo(x)
    loss_ipo = ipo_loss(logits_ipo, ref_logits, y_w, y_l, beta=0.1)
    opt_ipo.zero_grad()
    loss_ipo.backward()
    opt_ipo.step()

    logits_cdpo = policy_cdpo(x)
    loss_cdpo = cdpo_loss(logits_cdpo, ref_logits, y_w, y_l, beta=0.1, epsilon=0.1)
    opt_cdpo.zero_grad()
    loss_cdpo.backward()
    opt_cdpo.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch {epoch+1:3d}: DPO={loss_dpo.item():.4f}, '
              f'IPO={loss_ipo.item():.4f}, cDPO={loss_cdpo.item():.4f}')

print(f'\nKey: IPO is more conservative, cDPO handles noisy labels.')
print(f'Use cDPO when preference data quality is uncertain.')

## 3. 长度归一化与参考模型免费方法

DPO的一个已知问题是**长度偏差**：模型倾向于生成长回复，因为更长的序列通常有更高的log概率和。长度归一化可以缓解这个问题。

### 长度归一化DPO
将log概率除以序列长度，消除长度偏差：
$$\hat{r}(x,y) = \frac{\beta}{|y|} \log \frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)}$$

### RSO（Rejection Sampling Optimization）
结合拒绝采样和DPO，先采样多个回复，再用奖励模型排序，构建偏好对。

In [ ]:
def length_normalized_dpo_loss(policy_logits, ref_logits, y_chosen, y_rejected,
                                lengths_chosen, lengths_rejected, beta=0.1):
    log_pi_chosen = F.log_softmax(policy_logits, dim=-1).gather(1, y_chosen.unsqueeze(1)).squeeze(1)
    log_pi_rejected = F.log_softmax(policy_logits, dim=-1).gather(1, y_rejected.unsqueeze(1)).squeeze(1)
    log_ref_chosen = F.log_softmax(ref_logits, dim=-1).gather(1, y_chosen.unsqueeze(1)).squeeze(1)
    log_ref_rejected = F.log_softmax(ref_logits, dim=-1).gather(1, y_rejected.unsqueeze(1)).squeeze(1)

    ln_ratio_chosen = (log_pi_chosen - log_ref_chosen) / lengths_chosen.float()
    ln_ratio_rejected = (log_pi_rejected - log_ref_rejected) / lengths_rejected.float()

    loss = -F.logsigmoid(beta * (ln_ratio_chosen - ln_ratio_rejected)).mean()
    return loss

class RejectionSamplingOptimizer:
    def __init__(self, policy, reward_fn, n_samples=4, beta=0.1):
        self.policy = policy
        self.reward_fn = reward_fn
        self.n_samples = n_samples
        self.beta = beta

    @torch.no_grad()
    def generate_preference_pairs(self, x):
        vocab_size = self.policy.net[-1].out_features
        all_actions = []
        all_rewards = []
        for _ in range(self.n_samples):
            logits = self.policy(x)
            log_probs = F.log_softmax(logits, dim=-1)
            actions = log_probs.exp().multinomial(1).squeeze(-1)
            action_emb = F.one_hot(actions, vocab_size).float()
            rewards = self.reward_fn(action_emb)
            all_actions.append(actions)
            all_rewards.append(rewards)

        all_rewards = torch.stack(all_rewards)
        all_actions = torch.stack(all_actions)
        best_idx = all_rewards.argmax(dim=0)
        worst_idx = all_rewards.argmin(dim=0)
        batch_idx = torch.arange(x.shape[0])
        y_chosen = all_actions[best_idx, batch_idx]
        y_rejected = all_actions[worst_idx, batch_idx]
        return y_chosen, y_rejected

policy = SimplePolicy()
ref_policy = SimplePolicy()
ref_policy.load_state_dict(policy.state_dict())
for p in ref_policy.parameters():
    p.requires_grad = False

x = torch.randn(16, 64)
y_w = torch.randint(0, 50, (16,))
y_l = torch.randint(0, 50, (16,))
lengths_w = torch.randint(5, 20, (16,))
lengths_l = torch.randint(5, 20, (16,))

with torch.no_grad():
    ref_logits = ref_policy(x)
policy_logits = policy(x)

ln_loss = length_normalized_dpo_loss(policy_logits, ref_logits, y_w, y_l,
                                      lengths_w, lengths_l, beta=0.1)
std_loss, _ = dpo_loss(policy_logits, ref_logits, y_w, y_l, beta=0.1)

print('=== Length Normalization ===')
print(f'Standard DPO loss: {std_loss.item():.4f}')
print(f'Length-normalized DPO loss: {ln_loss.item():.4f}')
print(f'\nKey: Length normalization prevents the model from gaming reward via length.')

reward_fn = lambda emb: emb.sum(dim=-1)
rso = RejectionSamplingOptimizer(policy, reward_fn, n_samples=4)
y_chosen_rso, y_rejected_rso = rso.generate_preference_pairs(x)
print(f'\n=== Rejection Sampling ===')
print(f'Generated {x.shape[0]} preference pairs from {rso.n_samples} samples each')
print(f'Chosen actions: {y_chosen_rso[:8].tolist()}')
print(f'Rejected actions: {y_rejected_rso[:8].tolist()}')
print(f'\nKey: RSO combines sampling with reward-based selection for better pairs.')

## 4. DPO训练的完整Pipeline

工业级DPO训练需要考虑数据处理、训练调度、评估等环节。

In [ ]:
class DPOTrainer:
    def __init__(self, policy, ref_policy, beta=0.1, lr=1e-5,
                 weight_decay=0.01, max_grad_norm=1.0,
                 length_normalize=False, label_smoothing=0.0,
                 beta_schedule='constant', beta_min=0.05, warmup_steps=100):
        self.policy = policy
        self.ref_policy = ref_policy
        self.beta = beta
        self.length_normalize = length_normalize
        self.label_smoothing = label_smoothing
        self.beta_schedule = beta_schedule
        self.beta_min = beta_min
        self.warmup_steps = warmup_steps

        self.optimizer = torch.optim.AdamW(
            policy.parameters(), lr=lr, weight_decay=weight_decay,
            betas=(0.9, 0.95))
        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            self.optimizer, T_max=1000, eta_min=lr * 0.1)
        self.max_grad_norm = max_grad_norm
        self.metrics_history = []

    def get_beta(self, step):
        if self.beta_schedule == 'constant':
            return self.beta
        elif self.beta_schedule == 'linear':
            progress = min(step / self.warmup_steps, 1.0)
            return self.beta_min + (self.beta - self.beta_min) * (1 - progress)
        elif self.beta_schedule == 'cosine':
            progress = min(step / self.warmup_steps, 1.0)
            return self.beta_min + (self.beta - self.beta_min) * 0.5 * (1 + torch.cos(torch.tensor(progress * 3.14159))).item()
        return self.beta

    def train_step(self, x, y_chosen, y_rejected, lengths_chosen=None, lengths_rejected=None):
        current_beta = self.get_beta(len(self.metrics_history))

        policy_logits = self.policy(x)
        with torch.no_grad():
            ref_logits = self.ref_policy(x)

        log_pi_chosen = F.log_softmax(policy_logits, dim=-1).gather(1, y_chosen.unsqueeze(1)).squeeze(1)
        log_pi_rejected = F.log_softmax(policy_logits, dim=-1).gather(1, y_rejected.unsqueeze(1)).squeeze(1)
        log_ref_chosen = F.log_softmax(ref_logits, dim=-1).gather(1, y_chosen.unsqueeze(1)).squeeze(1)
        log_ref_rejected = F.log_softmax(ref_logits, dim=-1).gather(1, y_rejected.unsqueeze(1)).squeeze(1)

        log_ratio_chosen = log_pi_chosen - log_ref_chosen
        log_ratio_rejected = log_pi_rejected - log_ref_rejected

        if self.length_normalize and lengths_chosen is not None:
            log_ratio_chosen = log_ratio_chosen / lengths_chosen.float()
            log_ratio_rejected = log_ratio_rejected / lengths_rejected.float()

        diff = current_beta * (log_ratio_chosen - log_ratio_rejected)

        if self.label_smoothing > 0:
            loss_correct = -F.logsigmoid(diff)
            loss_flipped = -F.logsigmoid(-diff)
            loss = ((1 - self.label_smoothing) * loss_correct + self.label_smoothing * loss_flipped).mean()
        else:
            loss = -F.logsigmoid(diff).mean()

        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.policy.parameters(), self.max_grad_norm)
        self.optimizer.step()
        self.scheduler.step()

        with torch.no_grad():
            chosen_rewards = current_beta * log_ratio_chosen
            rejected_rewards = current_beta * log_ratio_rejected
            accuracy = (chosen_rewards > rejected_rewards).float().mean()
            margin = (chosen_rewards - rejected_rewards).mean()

        metrics = {
            'loss': loss.item(),
            'accuracy': accuracy.item(),
            'margin': margin.item(),
            'beta': current_beta,
            'lr': self.optimizer.param_groups[0]['lr'],
            'chosen_reward': chosen_rewards.mean().item(),
            'rejected_reward': rejected_rewards.mean().item(),
        }
        self.metrics_history.append(metrics)
        return metrics

policy = SimplePolicy()
ref_policy = SimplePolicy()
ref_policy.load_state_dict(policy.state_dict())
for p in ref_policy.parameters():
    p.requires_grad = False

trainer = DPOTrainer(policy, ref_policy, beta=0.1, lr=1e-3,
                      beta_schedule='cosine', label_smoothing=0.05)

x = torch.randn(32, 64)
y_w = torch.randint(0, 50, (32,))
y_l = torch.randint(0, 50, (32,))

print('=== DPO Training Pipeline ===')
for step in range(30):
    metrics = trainer.train_step(x, y_w, y_l)
    if (step + 1) % 10 == 0:
        print(f'Step {step+1:3d}: loss={metrics["loss"]:.4f}, '
              f'acc={metrics["accuracy"]:.2%}, margin={metrics["margin"]:.4f}, '
              f'beta={metrics["beta"]:.4f}, lr={metrics["lr"]:.2e}')

print(f'\nKey: Full DPO pipeline with beta scheduling, label smoothing, LR decay.')

## 5. DPO偏好数据构建

DPO的效果高度依赖偏好数据的质量。工业实践中，偏好数据的构建是最关键的环节。

### 偏好数据来源
1. **人工标注**：标注者对两个回复排序，质量最高但成本大
2. **AI标注（RLAIF）**：用强模型（GPT-4等）替代人工标注
3. **自动构建**：用模型采样多个回复，用规则或奖励模型排序

### 偏好数据质量控制
- **一致性过滤**：移除标注者不一致的数据
- **难度过滤**：移除过于简单的偏好对（chosen和rejected差距太大）
- **多样性保证**：确保覆盖不同任务类型和难度

In [ ]:
class PreferenceDataBuilder:
    def __init__(self, policy, reward_fn, n_samples=4, temperature=1.0):
        self.policy = policy
        self.reward_fn = reward_fn
        self.n_samples = n_samples
        self.temperature = temperature

    @torch.no_grad()
    def build_pairs(self, x, min_margin=0.1, max_margin=5.0):
        vocab_size = self.policy.net[-1].out_features
        all_actions = []
        all_rewards = []
        for _ in range(self.n_samples):
            logits = self.policy(x) / self.temperature
            log_probs = F.log_softmax(logits, dim=-1)
            actions = log_probs.exp().multinomial(1).squeeze(-1)
            action_emb = F.one_hot(actions, vocab_size).float()
            rewards = self.reward_fn(action_emb)
            all_actions.append(actions)
            all_rewards.append(rewards)

        all_rewards = torch.stack(all_rewards)
        all_actions = torch.stack(all_actions)

        sorted_idx = all_rewards.argsort(dim=0, descending=True)
        best_idx = sorted_idx[0]
        worst_idx = sorted_idx[-1]
        batch_idx = torch.arange(x.shape[0])

        y_chosen = all_actions[best_idx, batch_idx]
        y_rejected = all_actions[worst_idx, batch_idx]
        r_chosen = all_rewards[best_idx, batch_idx]
        r_rejected = all_rewards[worst_idx, batch_idx]

        margin = r_chosen - r_rejected
        valid = (margin > min_margin) & (margin < max_margin)

        return y_chosen[valid], y_rejected[valid], margin[valid]

    @staticmethod
    def filter_by_consistency(pairs, agreement_threshold=0.7):
        filtered = []
        for pair in pairs:
            n_annotators = 5
            agreements = torch.bernoulli(torch.tensor(agreement_threshold + 0.2),
                                         (n_annotators,)).sum() / n_annotators
            if agreements >= agreement_threshold:
                filtered.append(pair)
        return filtered

policy = SimplePolicy()
reward_fn = lambda emb: emb.sum(dim=-1) * 2 - 1
builder = PreferenceDataBuilder(policy, reward_fn, n_samples=8, temperature=0.8)

x = torch.randn(64, 64)
y_chosen, y_rejected, margins = builder.build_pairs(x, min_margin=0.05)

print('=== Preference Data Construction ===')
print(f'Input samples: {x.shape[0]}')
print(f'Valid preference pairs: {y_chosen.shape[0]}')
print(f'Filter rate: {(1 - y_chosen.shape[0]/x.shape[0]):.1%} removed by margin filter')
print(f'Margin stats: mean={margins.mean():.3f}, std={margins.std():.3f}, '
      f'min={margins.min():.3f}, max={margins.max():.3f}')

print(f'\nKey: Margin filtering removes trivial pairs and noisy pairs.')
print(f'More samples per prompt -> better pair quality -> better DPO performance.')

## 6. DPO工业实践要点

### 超参数选择
| 参数 | 推荐范围 | 说明 |
|------|----------|------|
| β (beta) | 0.05 - 0.5 | 控制偏离参考模型的程度，越大越保守 |
| 学习率 | 1e-6 ~ 5e-6 | 通常比SFT学习率小5-10倍 |
| 批量大小 | 128 - 1024 | 偏好对数量 |
| label_smoothing | 0.0 - 0.1 | 处理标注噪声 |
| max_grad_norm | 0.5 - 1.0 | 梯度裁剪 |

### β参数的影响
- **β太小**：策略几乎不变，对齐效果弱
- **β太大**：策略偏离参考模型太远，可能损害能力
- **推荐做法**：从β=0.1开始，根据验证集表现调整

### DPO vs RLHF选择指南
| 维度 | DPO | RLHF |
|------|-----|------|
| 显存需求 | 低（2个模型） | 高（4个模型） |
| 训练稳定性 | 高 | 低 |
| 在线探索 | 不支持 | 支持 |
| 偏好数据需求 | 必须 | 可在线收集 |
| 实现复杂度 | 低 | 高 |
| 上限性能 | 略低 | 略高 |

### 最佳实践
1. **先DPO后RLHF**：DPO做初步对齐，RLHF精调
2. **迭代DPO**：用对齐后的模型生成新的偏好数据，再训练
3. **数据质量 > 数据量**：1K高质量偏好对 > 10K低质量偏好对
4. **参考模型要冻结**：参考模型必须是SFT后的模型，不能更新
5. **监控隐式奖励**：跟踪$\beta \log(\pi_\theta/\pi_{ref})$，确保chosen > rejected